In [2]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
#from mpl_toolkits.axes_grid1.inset_locator import inset_axes

#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 52.63it/s]


Numba compilation complete!


In [3]:
#initial file processing
workcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = workcomp

#Comment depending on project type
#eopn3work = "N"
eopn3work = "Y"
eopn3work = "tgt"

if eopn3work == "Y":
    whomst ="NL"
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\2. Processed\\" + whomst + "\\"
    savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\3. Compiled\\"  + whomst + "\\"
    
if eopn3work == "tgt":
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\2. Processed\\" 
    savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\3. Compiled\\" 
    
else:
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
    savedir = titledpath + filedir + "Compilation with delta\\2025meandiffcollection\\"
    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "eOPN3"
respondercsv = responder + ".csv"
wt = "w1118"


In [4]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

# for file_no in os.listdir(openPath): 
#     if respondercsv in file_no and "w1118" not in file_no :   
#         f = os.path.join(openPath, file_no)
#         dfe=pd.read_csv(f)
#         exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
#         driver = file_no.split(" ")[0]
#         lstnew.append(driver)
# lst = lstnew.copy()

#processing ONLY specific names
lst = ["vGAT"]

print(lst)

['vGAT']


In [5]:
diff = pd.DataFrame()
diffbs = pd.DataFrame()

for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))
    
    #calculations
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True) 
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    dff2_prop = NLMATH.deltaversion(df_f, "binary_fallvalue", "fallprop")
    dff2_number = NLMATH.deltaversion(df_f, "Fall", "fallnumber")   #number of falls is under deltaversion_binary because the number of flies that fall could actually be so few in number, that the SD is 0, and thus hedges g will not be able to perform since the divisor ==0
    dfs2 = NLMATH.deltaversion(df_sp, "Velocity", "speed")
    dfh2 = NLMATH.deltaversion(df_h, "Y", "height")
    dfbs2 = NLMATH.deltaversion(df_bsp, "BSpeed", "bspeed")
    dfmv2 = NLMATH.deltaversion(df_maxv, "maxvelocity", "maxvelocity")

    dftotal = pd.concat([dff2_prop, dff2_number, dfs2,  dfh2, dfbs2,  dfmv2], axis = 1)
    dftotal['MBON'] = n

    dftotal.set_index("MBON", inplace = True)
    #dftotal.to_csv(savedir + n + " x " + responder + " allstats.csv")
    print(dftotal)


vGAT
      fallprop_bootstrap  fallprop_deltag  fallnumber_bootstrap  \
MBON                                                              
vGAT            2.052755            1.839              0.762904   
vGAT            1.862003            1.839              0.599349   
vGAT            1.754343            1.839              0.623042   
vGAT            1.720673            1.839              0.324850   
vGAT            1.814315            1.839              0.656692   
...                  ...              ...                   ...   
vGAT            2.133825            1.839              0.450801   
vGAT            1.828477            1.839              0.606694   
vGAT            1.667782            1.839              0.496139   
vGAT            1.936136            1.839              0.610048   
vGAT            1.960847            1.839              0.727030   

      fallnumber_deltag  speed_bootstrap  speed_deltag  height_bootstrap  \
MBON                                           